# AzLegalRAG - Colab Demo

Azerbaijani Legal Q&A using RAG with bge-m3 embeddings.

**Requirements**: T4 or L4 GPU

Go to: Runtime > Change runtime type > T4 GPU

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate sentence-transformers langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface chromadb datasets tqdm huggingface_hub

## 2. HuggingFace Login (Required for Gated Dataset)

The `allmalab/eqanun` dataset is gated. You need to:
1. Create a token at https://huggingface.co/settings/tokens
2. Accept access at https://huggingface.co/datasets/allmalab/eqanun
3. Run the cell below and paste your token

In [ ]:
from huggingface_hub import login
login()

## 3. Clone Repository

In [ ]:
!git clone https://github.com/StartZer0/AzLegalRAG.git
%cd AzLegalRAG

## 4. Check GPU

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 5. Ingest Documents (Run Once - Takes ~30 min)

This uses BAAI/bge-m3 (~2GB) for embeddings.

In [ ]:
import sys
sys.path.insert(0, './src')

from ingest import load_eqanun, chunk_documents

# Load dataset
dataset = load_eqanun()
print(f"Loaded {len(dataset)} documents")

# Chunk documents
chunks = chunk_documents(dataset)
print(f"Created {len(chunks)} chunks")

In [ ]:
# Create vectorstore with bge-m3 embeddings
from embed import create_vectorstore

vectorstore = create_vectorstore(chunks)
print("Vectorstore created!")

## 6. Test Search (Without LLM)

In [ ]:
from embed import load_vectorstore
from retrieve import semantic_search

vs = load_vectorstore()
results = semantic_search(vs, "Emek muqavilesi nedir?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n[{i}] Source: {doc.metadata['source']}")
    print(doc.page_content[:300] + "...")

## 7. Load LLM and Create RAG Chain

In [ ]:
from generate import get_llm, create_rag_chain
from embed import load_vectorstore

vs = load_vectorstore()
llm = get_llm()
chain = create_rag_chain(vs, llm)
print("RAG chain ready!")

## 8. Test RAG Q&A

In [ ]:
question = "Emek muqavilesi nedir?"
result = chain({"query": question})

print("=" * 50)
print(f"Question: {question}")
print("=" * 50)
print(f"\nAnswer:\n{result['result']}")
print("\n" + "=" * 50)
print("Sources:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

In [ ]:
# Try more questions
questions = [
    "Mehkeme qerari nece shekillendirilir?",
    "Nikah muqavilesi ucun ne teleb olunur?",
    "Vergiler hansi novlere bolunur?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = chain({"query": q})
    print(f"A: {result['result'][:500]}...")

## 9. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/AzLegalRAG
!cp -r ./vectorstore /content/drive/MyDrive/AzLegalRAG/
print("Vectorstore saved to Google Drive!")